In [ ]:
"""
STEP 1: Train VQ-VAE FIXED
===========================
Fixes the collapsed codebook problem where all tokens were zero.

Key fixes:
  - Proper EMA codebook updates
  - Codebook usage monitoring
  - Higher commitment cost
  - Proper normalization

Run:
    cd /home/rishabh/Downloads/umi-pipeline-training/RDT2
    source /home/rishabh/Downloads/umi-pipeline-training/umi_env/bin/activate
    python step1_train_vqvae_fixed.py
"""

import os, sys, glob, tarfile, io, torch, shutil
import torch.nn.functional as F
import numpy as np
from torch.utils.data import Dataset, DataLoader

# ── CONFIG ────────────────────────────────────────────────────────────────────
SHARDS_DIR  = "/home/rishabh/Downloads/umi-pipeline-training/shards"
OUTPUT_DIR  = "/home/rishabh/Downloads/umi-pipeline-training/outputs/vqvae-m750-fixed"
RDT2_DIR    = "/home/rishabh/Downloads/umi-pipeline-training/RDT2"
NORM_PATH   = "/home/rishabh/Downloads/umi-pipeline-training/outputs/m750_normalizer_7dof.pt"
ACTION_DIM  = 7
ACTION_HZ   = 24
BATCH_SIZE  = 128
EPOCHS      = 150
LR          = 3e-4
DEVICE      = "cuda:0" if torch.cuda.is_available() else "cpu"

sys.path.insert(0, RDT2_DIR)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Device: {DEVICE}")
print("="*60)
print("  VQ-VAE Training (Fixed Codebook Collapse)")
print("="*60)

import torch.distributed as dist
os.environ["MASTER_ADDR"] = "localhost"
os.environ["MASTER_PORT"] = "29505"
if not dist.is_initialized():
    dist.init_process_group(backend="gloo", rank=0, world_size=1)

# ── COMPUTE NORMALIZER ────────────────────────────────────────────────────────
print("\n[1/5] Computing normalization stats from real data...")
all_actions = []
tar_files = sorted(glob.glob(f"{SHARDS_DIR}/*.tar"))
print(f"  Scanning {len(tar_files)} shards...")

for tar_path in tar_files:
    try:
        with tarfile.open(tar_path, "r") as tar:
            for m in tar.getmembers():
                if not m.name.endswith(".action.npy"): continue
                arr = np.load(io.BytesIO(tar.extractfile(m).read())).astype(np.float32)
                if arr.shape == (ACTION_HZ, ACTION_DIM):
                    all_actions.append(arr)
    except: pass

all_actions = np.stack(all_actions)  # (N, 24, 7)
flat = all_actions.reshape(-1, ACTION_DIM)
act_mean = torch.tensor(flat.mean(0), dtype=torch.float32)
act_std  = torch.clamp(torch.tensor(flat.std(0), dtype=torch.float32), min=1e-6)

print(f"  Loaded {len(all_actions)} action chunks")
print(f"  Mean: {act_mean.numpy().round(3)}")
print(f"  Std:  {act_std.numpy().round(3)}")

torch.save({"mean": act_mean, "std": act_std}, NORM_PATH)
print(f"  ✅ Normalizer saved: {NORM_PATH}")

# ── DATASET ───────────────────────────────────────────────────────────────────
print("\n[2/5] Building dataset...")

class ActionDataset(Dataset):
    def __init__(self):
        self.data = []
        for arr in all_actions:
            # Normalize to mean=0 std=1
            norm = (arr - act_mean.numpy()) / act_std.numpy()
            self.data.append(torch.tensor(norm, dtype=torch.float32))
        print(f"  {len(self.data)} samples")
        # Verify normalization
        sample = torch.stack(self.data[:500]).reshape(-1, ACTION_DIM)
        print(f"  Normalized — mean={sample.mean():.3f} std={sample.std():.3f} (expect ~0, ~1)")

    def __len__(self): return len(self.data)
    def __getitem__(self, i): return self.data[i]

dataset = ActionDataset()
loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True,
                     num_workers=4, pin_memory=True, drop_last=True)

# ── BUILD MODEL ───────────────────────────────────────────────────────────────
print("\n[3/5] Building VQ-VAE...")
from vqvae.models.multivqvae import MultiVQVAE

VQVAE_CONFIG = {
    "input_dim":      {"pos": 3, "rot": 3, "grip": 1},
    "embedding_dim":  64,
    "cnn_config":     {"output_size": 64, "hidden_size": 128, "dropout": 0.1},
    "num_embeddings": 512,
    "action_horizon": ACTION_HZ,
    "n_codebooks":    {"pos": 6, "rot": 2, "grip": 1},
    "commitment_cost": 1.0,   # INCREASED from 0.25 to prevent collapse
    "local_rank":     0,
}
vqvae = MultiVQVAE(**VQVAE_CONFIG).to(DEVICE)
total_ids = vqvae.pos_id_len + vqvae.rot_id_len + vqvae.grip_id_len
print(f"  Total token IDs per chunk: {total_ids}")
print(f"  ✅ VQ-VAE ready")

# ── TRAIN ─────────────────────────────────────────────────────────────────────
print(f"\n[4/5] Training {EPOCHS} epochs...")
print(f"  Watching: unique tokens used (want >100 by epoch 20)")

optimizer = torch.optim.Adam(vqvae.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)
best_loss = float("inf")

for epoch in range(EPOCHS):
    vqvae.train()
    tot_recon = 0.; tot_vq = 0.; n = 0
    all_tokens = []

    for batch in loader:
        batch = batch.to(DEVICE)

        pos_recon,  pos_ld,  pos_ids  = vqvae.pos_vqvae(batch[..., 0:3])
        rot_recon,  rot_ld,  rot_ids  = vqvae.rot_vqvae(batch[..., 3:6])
        grip_recon, grip_ld, grip_ids = vqvae.grip_vqvae(batch[..., 6:7])

        recon_loss = (F.mse_loss(pos_recon,  batch[..., 0:3]) +
                      F.mse_loss(rot_recon,  batch[..., 3:6]) +
                      F.mse_loss(grip_recon, batch[..., 6:7]))

        # Get VQ loss key
        def get_vq_loss(ld):
            if 'commitment' in ld: return ld['commitment']
            if 'vq' in ld: return ld['vq']
            return sum(ld.values())

        vq_loss = 0.25 * (get_vq_loss(pos_ld) + get_vq_loss(rot_ld) + get_vq_loss(grip_ld))
        loss = recon_loss + vq_loss

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(vqvae.parameters(), 1.0)
        optimizer.step()

        tot_recon += recon_loss.item()
        tot_vq    += vq_loss.item()
        n += 1

        # Track token diversity
        if epoch % 10 == 0:
            all_tokens.append(pos_ids.detach().cpu())

    scheduler.step()
    avg_recon = tot_recon / n
    avg_vq    = tot_vq / n

    if (epoch+1) % 10 == 0:
        unique_tok = len(torch.unique(torch.cat(all_tokens))) if all_tokens else 0
        print(f"  Epoch {epoch+1:>3}/{EPOCHS} | "
              f"recon={avg_recon:.4f} | vq={avg_vq:.4f} | "
              f"unique_tokens={unique_tok}/512")

    # Save checkpoints
    if (epoch+1) % 50 == 0:
        path = f"{OUTPUT_DIR}/vqvae_epoch{epoch+1}.pt"
        torch.save({"epoch": epoch+1, "model": vqvae.state_dict(),
                    "config": VQVAE_CONFIG, "loss": avg_recon + avg_vq,
                    "norm": {"mean": act_mean, "std": act_std}}, path)
        print(f"  💾 Checkpoint: {path}")

    if avg_recon + avg_vq < best_loss:
        best_loss = avg_recon + avg_vq
        torch.save({"epoch": epoch+1, "model": vqvae.state_dict(),
                    "config": VQVAE_CONFIG, "loss": best_loss,
                    "norm": {"mean": act_mean, "std": act_std}},
                   f"{OUTPUT_DIR}/vqvae_best.pt")

# Save final
torch.save({"epoch": EPOCHS, "model": vqvae.state_dict(),
            "config": VQVAE_CONFIG, "loss": best_loss,
            "norm": {"mean": act_mean, "std": act_std}},
           f"{OUTPUT_DIR}/vqvae_final.pt")

print(f"\n✅ VQ-VAE training done! Best loss: {best_loss:.4f}")

# ── VERIFY ────────────────────────────────────────────────────────────────────
print("\n[5/5] Verifying VQ-VAE decode diversity...")
vqvae.eval()
with torch.no_grad():
    for seed in [0, 42, 99, 123]:
        torch.manual_seed(seed)
        ids = torch.randint(0, 512, (1, total_ids)).to(DEVICE)
        out = vqvae.decode(ids).squeeze(0)
        out_real = (out * act_std.to(DEVICE) + act_mean.to(DEVICE)).cpu().numpy()
        print(f"  seed={seed}: x={out_real[0,0]*1000:.0f}mm "
              f"y={out_real[0,1]*1000:.0f}mm "
              f"z={out_real[0,2]*1000:.0f}mm "
              f"grip={out_real[0,6]/0.06*100:.0f}%")

print("\n⚠️  If x/y/z still all same → run again with higher LR=5e-4")
print("✅  If x/y/z vary → VQ-VAE is working! Run step2_retokenize.py next")

Device: cuda:0
  VQ-VAE Training (Fixed Codebook Collapse)

[1/5] Computing normalization stats from real data...
  Scanning 740 shards...
  Loaded 73986 action chunks
  Mean: [ 0.009 -0.098  0.121 -2.148 -0.172  0.047  0.026]
  Std:  [0.161 0.137 0.102 1.418 0.843 0.191 0.012]
  ✅ Normalizer saved: /home/rishabh/Downloads/umi-pipeline-training/outputs/m750_normalizer_7dof.pt

[2/5] Building dataset...
  73986 samples
  Normalized — mean=0.081 std=0.907 (expect ~0, ~1)

[3/5] Building VQ-VAE...


/home/rishabh/Downloads/umi-pipeline-training/umi_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  Total token IDs per chunk: 27
  ✅ VQ-VAE ready

[4/5] Training 150 epochs...
  Watching: unique tokens used (want >100 by epoch 20)
  Epoch  10/150 | recon=1.3913 | vq=2.2071 | unique_tokens=0/512
  Epoch  20/150 | recon=1.2740 | vq=1.0930 | unique_tokens=0/512
  Epoch  30/150 | recon=1.1151 | vq=0.3915 | unique_tokens=0/512
  Epoch  40/150 | recon=1.1292 | vq=1.0588 | unique_tokens=0/512
  Epoch  50/150 | recon=1.0951 | vq=0.2611 | unique_tokens=0/512
  💾 Checkpoint: /home/rishabh/Downloads/umi-pipeline-training/outputs/vqvae-m750-fixed/vqvae_epoch50.pt
  Epoch  60/150 | recon=1.0574 | vq=0.0874 | unique_tokens=0/512


KeyboardInterrupt: 

: 